# View PNML Models of Each Fragment

**Prerequisite:** Notebook `02_quality_measurments.ipynb` must have been run first.
That creates the metrics file, the XES sub-log files, and the PNML model files per fragment.

This notebook loads the pre-exported PNML files and displays the Petri net diagram for each fragment's process model.

In [ ]:
%load_ext autoreload
%autoreload 2
import sys
import os
sys.path.append('../')

from IPython.display import display, HTML

import pm4py
from pm4py.visualization.petri_net import visualizer as pn_visualizer

from process_fragment_miner.utils import (
    parse_metrics_file,
    get_xes_dir_and_base,
)

## 1. Configuration

In [ ]:
# --- CONFIG ---
METRICS_PATH = "../data/processed/BPIC/BPIC12.xes.gz.pfm.metrics.txt"
PNML_DIR = "../data/processed/BPIC/pnml"

# Methods to visualize (subset of: heuristic, bigram, similarity, frequency)
METHODS = ["heuristic"]

# Derive filename base from the metrics path
_, FILENAME_BASE = get_xes_dir_and_base(METRICS_PATH)
print(f"PNML dir: {PNML_DIR}")
print(f"Base:     {FILENAME_BASE}")

## 2. Get Fragments per Method

Loads fragment activity lists from the metrics file.

In [ ]:
fragments_by_method = {}

if os.path.exists(METRICS_PATH):
    parsed = parse_metrics_file(METRICS_PATH)
    for m in METHODS:
        if m in parsed:
            fragments_by_method[m] = parsed[m]
    print(f"Loaded from: {METRICS_PATH}")
    for m, data in fragments_by_method.items():
        print(f"  {m}: score={data['score']:.4f}, {len(data['fragments'])} fragments")
else:
    print(f"Not found: {METRICS_PATH}")
    print("Run 02_quality_measurments.ipynb first.")

if not fragments_by_method:
    print("No methods available.")

## 3. View Petri Net for Each Fragment

Loads the pre-exported PNML model file for each fragment and displays its Petri net diagram.

In [ ]:
for method, data in fragments_by_method.items():
    display(HTML(f'<h2>Method: {method} (score={data["score"]:.4f})</h2>'))

    for idx, activities in enumerate(data['fragments']):
        display(HTML(f'<h3>Fragment {idx} ({len(activities)} activities)</h3>'))
        display(HTML(f'<b>Activities:</b> {", ".join(activities)}'))

        # Locate the PNML file for this fragment
        pnml_path = os.path.join(PNML_DIR, f'{FILENAME_BASE}.{idx}.{method}.pnml')
        if not os.path.exists(pnml_path):
            print(f"  PNML not found: {pnml_path}")
            continue

        display(HTML(f'<i>Model:</i> {os.path.basename(pnml_path)}'))

        try:
            net, im, fm = pm4py.read_pnml(pnml_path)
            gviz = pn_visualizer.apply(net, im, fm)
            pn_visualizer.view(gviz)
        except Exception as e:
            print(f"  Error loading PNML: {e}")

        display(HTML('<hr>'))

    # Also show the root model if it exists
    root_path = os.path.join(PNML_DIR, f'{FILENAME_BASE}.root.{method}.pnml')
    if os.path.exists(root_path):
        display(HTML(f'<h2>Root Model ({method})</h2>'))
        display(HTML(f'<i>Model:</i> {os.path.basename(root_path)}'))
        try:
            net, im, fm = pm4py.read_pnml(root_path)
            gviz = pn_visualizer.apply(net, im, fm)
            pn_visualizer.view(gviz)
        except Exception as e:
            print(f"  Error loading root PNML: {e}")